Load, validate, clean, label and split the Canvas Network Person-Course data

In [49]:
import json
from pathlib import Path

import pandas as pd
import yaml
from sklearn.model_selection import train_test_split



 1. Define the features and required columns

In [50]:
FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
SOURCE_COL = "completed_%"
TARGET = "Engagement_Level"
REQUIRED = FEATURES + [SOURCE_COL, "userid_DI", "course_id_DI"]

- Feature descriptions:
- nevents: Number of distinct course interactions.
- ndays_act: Number of distinct days with course activity.
- ncontent: Percentage of unique course modules viewed.
- nforum_posts: Number of discussion-forum posts.
- completed_%: Course completion percentage used to derive the target.
- Engagement_Level: Binary target label (0 = low, 1 = high).
-userid_DI and course_id_DI: Required identifiers for schema validation.


 2. Load the configuration file

In [51]:
def find_project_root(start=None):
    """Walk upwards from `start` until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(
        f"Could not find configs/base.yaml above {start}"
    )


PROJECT_ROOT = find_project_root()

In [52]:
def resolve(path):
    """Turn a relative path from the config into an absolute path from the project root."""
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path

In [53]:
def load_config(path="configs/base.yaml"):
    """Load project settings from a YAML configuration file."""
    with open(resolve(path), "r", encoding="utf-8") as f:
        return yaml.safe_load(f)



3. Load the raw dataset

In [54]:
def load_raw(path):
    """Load the tab-separated Canvas dataset."""
    return pd.read_csv(resolve(path), sep="\t", low_memory=False)



4. Validate the dataset schema

In [55]:
def validate_schema(df):
    """Check that all required columns exist."""
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    

 5. Clean the data

In [56]:
def clean(df):
    """Keep rows with a non-missing completed_% value."""
    validate_schema(df)

    out = df[df[SOURCE_COL].notna()].reset_index(drop=True)

    # completed_% must be a 0-1 fraction, since the threshold is 0.5
    low, high = out[SOURCE_COL].min(), out[SOURCE_COL].max()
    if low < 0 or high > 1:
        raise ValueError(f"{SOURCE_COL} outside 0-1 (min={low}, max={high})")

    print(f"Cleaning: {len(df)} -> {len(out)} rows")
    return out


6. Create the engagement target

In [57]:
def make_target(df, threshold=0.5):
    """Create a binary engagement label from completed_%."""
    out = df.copy()
    out[TARGET] = (out[SOURCE_COL] >= threshold).astype(int)
    return out

7. Prevent target leakage

In [58]:
def assert_no_leakage(feature_cols):
    """Prevent target information from being used as input features."""
    leaked = {SOURCE_COL, TARGET}.intersection(feature_cols)
    if leaked:
        raise ValueError(f"Target leakage: {leaked} used as feature")


8. Build the model table

In [ ]:
def model_table(df):
    """Keep only the model features and the engagement target."""
    assert_no_leakage(FEATURES)
    return df[FEATURES + [TARGET]].copy()


 9. Split into training and testing sets

In [59]:
def split(df, test_size, seed):
    """Create a stratified, reproducible train/test split."""
    train, test = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[TARGET],
    )
    return train.reset_index(drop=True), test.reset_index(drop=True)




10. Run the complete data-preparation pipeline


In [61]:
def build_audit(raw, table, train, test, cfg):
    """Collect measured dataset statistics."""
    return {
        "raw_rows": int(len(raw)),
        "clean_rows": int(len(table)),
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "seed": cfg["seed"],
        "threshold": cfg["target_threshold"],
        "missing_pct_train": (train[FEATURES].isna().mean() * 100).round(2).to_dict(),
        "missing_pct_test": (test[FEATURES].isna().mean() * 100).round(2).to_dict(),
        "class_share_train": train[TARGET].value_counts(normalize=True).round(4).to_dict(),
        "class_share_test": test[TARGET].value_counts(normalize=True).round(4).to_dict(),
    }

In [62]:
def prepare(config_path="configs/base.yaml"):
    """Prepare datasets, save outputs and generate a data audit."""
    cfg = load_config(config_path)
    raw = load_raw(cfg["raw_path"])

    table = model_table(make_target(clean(raw), cfg["target_threshold"]))
    train, test = split(table, cfg["test_size"], cfg["seed"])

    out_dir = resolve(cfg["processed_dir"])
    res_dir = resolve(cfg["results_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    res_dir.mkdir(parents=True, exist_ok=True)

    train.to_csv(out_dir / "train.csv", index=False)
    test.to_csv(out_dir / "test.csv", index=False)

    audit = build_audit(raw, table, train, test, cfg)
    with open(res_dir / "data_audit.json", "w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2)

    return train, test, audit

In [63]:
print("Project root:", PROJECT_ROOT)
train, test, audit = prepare()
print(json.dumps(audit, indent=2))

Project root: D:\student_engagement_twin
Cleaning: 325199 -> 26661 rows
{
  "raw_rows": 325199,
  "clean_rows": 26661,
  "train_rows": 21328,
  "test_rows": 5333,
  "seed": 42,
  "threshold": 0.5,
  "missing_pct_train": {
    "nevents": 17.08,
    "ndays_act": 10.53,
    "ncontent": 0.0,
    "nforum_posts": 68.11
  },
  "missing_pct_test": {
    "nevents": 17.1,
    "ndays_act": 10.09,
    "ncontent": 0.0,
    "nforum_posts": 67.84
  },
  "class_share_train": {
    "0": 0.8246,
    "1": 0.1754
  },
  "class_share_test": {
    "0": 0.8247,
    "1": 0.1753
  }
}
